# Compiler Design Lab — Experiment 6: Lexical Analysis using Lex / Flex (Google Colab)


## 1. Installing Flex in Colab


In [1]:
!sudo apt-get update -y > /dev/null
!sudo apt-get install -y flex bison > /dev/null
!echo "Install step finished."


Install step finished.


In [2]:
!flex --version
!bison --version | head -1
!gcc --version | head -1


flex 2.6.4
bison (GNU Bison) 3.8.2
gcc (Ubuntu 15.2.0-16ubuntu1) 15.2.0


### Sample test — is Flex working?


In [3]:
%%writefile test1.l
%{
#include <stdio.h>
%}

%%
"hello"     { printf("Greeting found: %s\n", yytext); }
"world"     { printf("Place found: %s\n", yytext); }
\n          { /* ignore */ }
.           { /* ignore */ }
%%

int yywrap(void) { return 1; }
int main(void) { yylex(); return 0; }


Writing test1.l


In [4]:
!flex test1.l
!gcc lex.yy.c -o test -lfl
!echo "hello world, this is flex" | ./test


Greeting found: hello
Place found: world


## 2. Example Programs (building blocks for Exp 6)


### Example 1 — Minimal Flex program


In [5]:
%%writefile ex1_hello.l
%{
/* Minimal: recognise hello/world */
#include <stdio.h>
%}
%%
"hello"     { printf("Greeting found: %s\n", yytext); }
"world"     { printf("Place found: %s\n", yytext); }
\n          { }
.           { }
%%
int yywrap(void) { return 1; }
int main(void) { yylex(); return 0; }


Writing ex1_hello.l


In [6]:
!flex ex1_hello.l
!gcc lex.yy.c -o ex1_hello -lfl
!echo "hello world, this is flex" | ./ex1_hello


Greeting found: hello
Place found: world


### Example 2 — Keywords vs. identifiers (rule order matters)


In [7]:
%%writefile ex2_keyword.l
%{
#include <stdio.h>
%}
DIGIT   [0-9]
LETTER  [a-zA-Z_]
%%
"int"|"float"|"char"|"double"|"void"|"return"|"if"|"else"|"while"|"for"   { printf("%-12s KEYWORD\n", yytext); }
{LETTER}({LETTER}|{DIGIT})*   { printf("%-12s IDENTIFIER\n", yytext); }
[ \t\n]     { }
.           { }
%%
int yywrap(void) { return 1; }
int main(void) { yylex(); return 0; }


Writing ex2_keyword.l


In [8]:
!flex ex2_keyword.l
!gcc lex.yy.c -o ex2_keyword -lfl
!echo "int num1, num2, sum; float avg;" | ./ex2_keyword


int          KEYWORD
num1         IDENTIFIER
num2         IDENTIFIER
sum          IDENTIFIER
float        KEYWORD
avg          IDENTIFIER


### Example 3 — Integer vs. floating-point constants


In [9]:
%%writefile ex3_numbers.l
%{
#include <stdio.h>
%}
%%
[0-9]+\.[0-9]+   { printf("%-12s FLOAT_CONST\n", yytext); }
[0-9]+           { printf("%-12s INT_CONST\n", yytext); }
[ \t\n]         { }
.               { }
%%
int yywrap(void) { return 1; }
int main(void) { yylex(); return 0; }


Writing ex3_numbers.l


In [10]:
!flex ex3_numbers.l
!gcc lex.yy.c -o ex3_numbers -lfl
!echo "42 3.14 0 100.0 hello" | ./ex3_numbers


42           INT_CONST
3.14         FLOAT_CONST
0            INT_CONST
100.0        FLOAT_CONST


### Example 4 — Ignoring comments & whitespace (start conditions)


In [11]:
%%writefile ex4_comments.l
%{
#include <stdio.h>
%}
%x LINE_COMMENT BLOCK_COMMENT
%%
"//"                    { BEGIN(LINE_COMMENT); }
<LINE_COMMENT>\n        { BEGIN(INITIAL); putchar('\n'); }
<LINE_COMMENT>.         { }
"/*"                    { BEGIN(BLOCK_COMMENT); }
<BLOCK_COMMENT>"*/"     { BEGIN(INITIAL); }
<BLOCK_COMMENT>\n       { putchar('\n'); }
<BLOCK_COMMENT>.        { }
[ \t\n]+                { if (yytext[0]=='\n') putchar('\n'); }
.                       { putchar(yytext[0]); }
%%
int yywrap(void) { return 1; }
int main(void) { yylex(); return 0; }


Writing ex4_comments.l


In [12]:
%%writefile sample.c
#include <stdio.h>   // header
/* block
   comment */
int main(void) {
    printf("Hello"); // greet
    return 0;
}


Writing sample.c


In [13]:
!flex ex4_comments.l
!gcc lex.yy.c -o ex4_comments -lfl
!./ex4_comments < sample.c


#include<stdio.h>


intmain(void){
printf("Hello");
return0;
}


### Example 5 — Operators (arithmetic, relational, logical, assignment)


In [14]:
%%writefile ex5_operators.l
%{
#include <stdio.h>
%}
%%
"++"|"--"                         { printf("%-12s ARITH_OP\n", yytext); }
"+"|"-"|"*"|"/"|"%"               { printf("%-12s ARITH_OP\n", yytext); }
"=="|"!="|"<="|">="|"<"|">"       { printf("%-12s REL_OP\n", yytext); }
"&&"|"||"|"!"                      { printf("%-12s LOGICAL_OP\n", yytext); }
"+="|"-="|"*="|"/="|"%="|"="       { printf("%-12s ASSIGN_OP\n", yytext); }
[ \t\n]                             { }
.                                   { }
%%
int yywrap(void) { return 1; }
int main(void) { yylex(); return 0; }


Writing ex5_operators.l


In [15]:
!flex ex5_operators.l
!gcc lex.yy.c -o ex5_operators -lfl
!echo "a += b * c; if (x <= 10 && y != 0) x++;" | ./ex5_operators


+=           ASSIGN_OP
*            ARITH_OP
<=           REL_OP
&&           LOGICAL_OP
!=           REL_OP
++           ARITH_OP


### Example 6 — Delimiters


In [16]:
%%writefile ex6_delim.l
%{
#include <stdio.h>
%}
%%
","|";"|"{"|"}"|"("|")"|"["|"]"   { printf("%-12s DELIMITER\n", yytext); }
[ \t\n]                               { }
.                                     { }
%%
int yywrap(void) { return 1; }
int main(void) { yylex(); return 0; }


Writing ex6_delim.l


In [17]:
!flex ex6_delim.l
!gcc lex.yy.c -o ex6_delim -lfl
!echo "int f(int a, int b) { return (a + b); }" | ./ex6_delim


(            DELIMITER
,            DELIMITER
)            DELIMITER
{            DELIMITER
(            DELIMITER
)            DELIMITER
;            DELIMITER
}            DELIMITER


### Example 7 — Counting tokens (preview of Exp 6(c))


In [18]:
%%writefile ex7_counter.l
%{
#include <stdio.h>
int kw=0, id=0, nums=0;
%}
DIGIT   [0-9]
LETTER  [a-zA-Z_]
%%
"int"|"float"|"char"|"double"|"void"|"return"|"if"|"else"|"while"|"for"   { kw++; }
{LETTER}({LETTER}|{DIGIT})*         { id++; }
[0-9]+(\.[0-9]+)?                  { nums++; }
[ \t\n]                           { }
.                                 { }
%%
int yywrap(void) { return 1; }
int main(void) { yylex(); printf("Keywords: %d\nIdentifiers: %d\nNumbers: %d\n", kw, id, nums); return 0; }


Writing ex7_counter.l


In [19]:
!flex ex7_counter.l
!gcc lex.yy.c -o ex7_counter -lfl
!echo "int a = 10; float b = 3.14;" | ./ex7_counter


Keywords: 2
Identifiers: 2
Numbers: 2


## 3. Experiment 6 — Lab Programs (6a, 6b, 6c)


### 6(a) — Keywords, identifiers, integer & float constants (ignore comments/whitespace)


In [20]:
%%writefile exp6a.l
%{
/* Exp 6(a): keywords, identifiers, integer & float constants */
#include <stdio.h>
%}
%x LINE_COMMENT BLOCK_COMMENT
DIGIT   [0-9]
LETTER  [a-zA-Z_]
%%
"//"                    { BEGIN(LINE_COMMENT); }
<LINE_COMMENT>\n        { BEGIN(INITIAL); }
<LINE_COMMENT>.         { /* skip */ }
"/*"                    { BEGIN(BLOCK_COMMENT); }
<BLOCK_COMMENT>"*/"     { BEGIN(INITIAL); }
<BLOCK_COMMENT>\n       { /* skip */ }
<BLOCK_COMMENT>.        { /* skip */ }
[ \t\n]+                { /* skip whitespace */ }
"auto"|"break"|"case"|"char"|"const"|"continue"|"default"|"do"|"double"|"else"|"enum"|"extern"|"float"|"for"|"goto"|"if"|"int"|"long"|"register"|"return"|"short"|"signed"|"sizeof"|"static"|"struct"|"switch"|"typedef"|"union"|"unsigned"|"void"|"volatile"|"while"   { printf("%-20s KEYWORD\n", yytext); }
{LETTER}({LETTER}|{DIGIT})*   { printf("%-20s IDENTIFIER\n", yytext); }
[0-9]+\.[0-9]+               { printf("%-20s FLOAT_CONST\n", yytext); }
[0-9]+                       { printf("%-20s INT_CONST\n", yytext); }
.                           { /* ignore operators/delimiters for this part */ }
%%
int yywrap(void) { return 1; }
int main(int argc, char *argv[]) {
    if (argc > 1) { yyin = fopen(argv[1], "r"); if (!yyin) { perror(argv[1]); return 1; } }
    yylex();
    return 0;
}


Writing exp6a.l


In [21]:
%%writefile input6a.c
#include <stdio.h>
// simple program with comments
/* block comment
   spanning lines */
int main(void) {
    int count = 10;
    float avg = 3.14;
    return 0;
}


Writing input6a.c


In [22]:
!flex exp6a.l
!gcc lex.yy.c -o exp6a -lfl
!./exp6a input6a.c


include              IDENTIFIER
stdio                IDENTIFIER
h                    IDENTIFIER
int                  KEYWORD
main                 IDENTIFIER
void                 KEYWORD
int                  KEYWORD
count                IDENTIFIER
10                   INT_CONST
float                KEYWORD
avg                  IDENTIFIER
3.14                 FLOAT_CONST
return               KEYWORD
0                    INT_CONST


### 6(b) — Operators & delimiters


In [23]:
%%writefile exp6b.l
%{
/* Exp 6(b): operators and delimiters */
#include <stdio.h>
%}
%x LINE_COMMENT BLOCK_COMMENT
%%
"//"                    { BEGIN(LINE_COMMENT); }
<LINE_COMMENT>\n        { BEGIN(INITIAL); }
<LINE_COMMENT>.         { }
"/*"                    { BEGIN(BLOCK_COMMENT); }
<BLOCK_COMMENT>"*/"     { BEGIN(INITIAL); }
<BLOCK_COMMENT>.        { }
<BLOCK_COMMENT>\n       { }
[ \t\n]+                { /* skip whitespace */ }
"++"|"--"                         { printf("%-12s ARITHMETIC_OP\n", yytext); }
"+"|"-"|"*"|"/"|"%"               { printf("%-12s ARITHMETIC_OP\n", yytext); }
"=="|"!="|"<="|">="                { printf("%-12s RELATIONAL_OP\n", yytext); }
"<"|">"                           { printf("%-12s RELATIONAL_OP\n", yytext); }
"&&"|"||"                         { printf("%-12s LOGICAL_OP\n", yytext); }
"!"                               { printf("%-12s LOGICAL_OP\n", yytext); }
"+="|"-="|"*="|"/="|"%="           { printf("%-12s ASSIGNMENT_OP\n", yytext); }
"="                               { printf("%-12s ASSIGNMENT_OP\n", yytext); }
","|";"|"{"|"}"|"("|")"|"["|"]"   { printf("%-12s DELIMITER\n", yytext); }
.                               { /* ignore identifiers/numbers for this part */ }
%%
int yywrap(void) { return 1; }
int main(int argc, char *argv[]) {
    if (argc > 1) { yyin = fopen(argv[1], "r"); if (!yyin) { perror(argv[1]); return 1; } }
    yylex();
    return 0;
}


Writing exp6b.l


In [24]:
%%writefile input6b.c
int main(void) {
    int a = 10, b = 20;
    if (a <= b && b != 0) {
        a += b * 2;
        a++;
    }
    return 0;
}


Writing input6b.c


In [25]:
!flex exp6b.l
!gcc lex.yy.c -o exp6b -lfl
!./exp6b input6b.c


(            DELIMITER
)            DELIMITER
{            DELIMITER
=            ASSIGNMENT_OP
,            DELIMITER
=            ASSIGNMENT_OP
;            DELIMITER
(            DELIMITER
<=           RELATIONAL_OP
&&           LOGICAL_OP
!=           RELATIONAL_OP
)            DELIMITER
{            DELIMITER
+=           ASSIGNMENT_OP
*            ARITHMETIC_OP
;            DELIMITER
++           ARITHMETIC_OP
;            DELIMITER
}            DELIMITER
;            DELIMITER
}            DELIMITER


### 6(c) — Count keywords, identifiers, numeric constants, operators, delimiters, lines


In [26]:
%%writefile exp6c.l
%{
/* Exp 6(c): count keywords, identifiers, numbers, operators, delimiters, lines */
#include <stdio.h>
int kw=0, id=0, num=0, op=0, delim=0, lines=0;
%}
%x LINE_COMMENT BLOCK_COMMENT
DIGIT   [0-9]
LETTER  [a-zA-Z_]
%%
"//"                    { BEGIN(LINE_COMMENT); }
<LINE_COMMENT>\n        { lines++; BEGIN(INITIAL); }
<LINE_COMMENT>.         { }
"/*"                    { BEGIN(BLOCK_COMMENT); }
<BLOCK_COMMENT>"*/"     { BEGIN(INITIAL); }
<BLOCK_COMMENT>\n       { lines++; }
<BLOCK_COMMENT>.        { }
\n                     { lines++; }
[ \t\r]+                { /* skip */ }
"auto"|"break"|"case"|"char"|"const"|"continue"|"default"|"do"|"double"|"else"|"enum"|"extern"|"float"|"for"|"goto"|"if"|"int"|"long"|"register"|"return"|"short"|"signed"|"sizeof"|"static"|"struct"|"switch"|"typedef"|"union"|"unsigned"|"void"|"volatile"|"while"   { kw++; }
{LETTER}({LETTER}|{DIGIT})*   { id++; }
[0-9]+\.[0-9]+               { num++; }
[0-9]+                       { num++; }
"++"|"--"|"+"|"-"|"*"|"/"|"%"|"=="|"!="|"<="|">="|"<"|">"|"&&"|"||"|"!"|"="|"+="|"-="|"*="|"/="|"%="   { op++; }
","|";"|"{"|"}"|"("|")"|"["|"]"   { delim++; }
.                           { /* ignore string literals etc. */ }
%%
int yywrap(void) { return 1; }
int main(int argc, char *argv[]) {
    if (argc > 1) { yyin = fopen(argv[1], "r"); if (!yyin) { perror(argv[1]); return 1; } }
    yylex();
    printf("Keywords         : %d\n", kw);
    printf("Identifiers      : %d\n", id);
    printf("Numeric constants: %d\n", num);
    printf("Operators        : %d\n", op);
    printf("Delimiters       : %d\n", delim);
    printf("Lines            : %d\n", lines);
    return 0;
}


Writing exp6c.l


In [27]:
%%writefile input6c.c
#include <stdio.h>
int main(void) {
    int a = 10, b = 20;
    float avg = (a + b) / 2.0;
    if (avg >= 15 && a != 0) {
        avg += 1;
    }
    return 0;
}


Writing input6c.c


In [28]:
!flex exp6c.l
!gcc lex.yy.c -o exp6c -lfl
!./exp6c input6c.c


Keywords         : 6
Identifiers      : 12
Numeric constants: 7
Operators        : 11
Delimiters       : 15
Lines            : 10
